In [2]:
import pandas as pd
import numpy as np

# Load the cleaned dataset
df = pd.read_csv(r"C:\Users\rchan\Downloads\data\processed\Cleaned_data.csv")

# Combine Date and Time into a single Datetime object
df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], dayfirst=True)

# 1. Cross-Border Transaction Flag
df['is_cross_border'] = (df['Sender_bank_location'] != df['Receiver_bank_location']).astype(int)

# 2. Currency Mismatch Feature
df['currency_mismatch'] = (df['Payment_currency'] != df['Received_currency']).astype(int)

# 3. Transaction Hour
df['transaction_hour'] = df['Datetime'].dt.hour

# 4. Transaction Day of Week
df['day_of_week'] = df['Datetime'].dt.dayofweek

# 5. Sender Transaction Count
df['sender_txn_count'] = df.groupby('Sender_account')['Amount'].transform('count')

# 6. Receiver Transaction Count
df['receiver_txn_count'] = df.groupby('Receiver_account')['Amount'].transform('count')

# 7. Average Sender Transaction Amount
df['sender_avg_amount'] = df.groupby('Sender_account')['Amount'].transform('mean')

# 8. Transaction Amount Deviation
df['amount_deviation'] = df['Amount'] - df['sender_avg_amount']
df['amount_deviation_ratio'] = df['Amount'] / (df['sender_avg_amount'] + 1e-5) # Added small epsilon to avoid division by zero

# 9. Country Risk Score 
# We simulate risk based on frequency. Rare countries = High Risk (3), Moderate = Medium (2), Common = Low (1)
loc_freq = df['Sender_bank_location'].value_counts(normalize=True)
def assign_risk(freq):
    if freq < 0.05: return 3
    elif freq < 0.20: return 2
    else: return 1

df['country_risk_sender'] = df['Sender_bank_location'].map(loc_freq).apply(assign_risk)
loc_freq_rec = df['Receiver_bank_location'].value_counts(normalize=True)
df['country_risk_receiver'] = df['Receiver_bank_location'].map(loc_freq_rec).apply(assign_risk)
df['country_risk_score'] = df['country_risk_sender'] + df['country_risk_receiver']

# 10. Sender-Receiver Relationship Count
df['sender_receiver_txn_count'] = df.groupby(['Sender_account', 'Receiver_account'])['Amount'].transform('count')

# 11. Transaction Velocity (Last 24 hours per sender)
# Sort by Sender and Datetime for rolling window calculation
df = df.sort_values(by=['Sender_account', 'Datetime'])
df = df.set_index('Datetime')
df['dummy'] = 1
df['transaction_velocity_24h'] = df.groupby('Sender_account')['dummy'].rolling('24h').count().reset_index(level=0, drop=True)
df = df.drop(columns=['dummy']).reset_index()

# 12. Currency Risk Feature (Simulated using frequency similarly to country risk)
curr_freq = df['Payment_currency'].value_counts(normalize=True)
df['currency_risk'] = df['Payment_currency'].map(curr_freq).apply(assign_risk)

# Flags
# Feature 1 — Amount risk
df["high_amount_flag"] = (df["Amount"] > df["sender_avg_amount"] * 3).astype(int)

# Feature 2 — New receiver risk (0 previous transactions between this sender and receiver before this row)
df = df.sort_values(by=['Sender_account', 'Receiver_account', 'Datetime'])
df['past_sender_receiver_count'] = df.groupby(['Sender_account', 'Receiver_account']).cumcount()
df["new_receiver"] = (df['past_sender_receiver_count'] == 0).astype(int)

# Feature 3 — Night transaction risk
df["night_transaction"] = ((df["transaction_hour"] >= 0) & (df["transaction_hour"] <= 5)).astype(int)

# Feature 4 — High velocity risk
df["velocity_flag"] = (df["transaction_velocity_24h"] > 10).astype(int)

# --- ADDITIONAL FEATURES ---
# Extra 1: Receiver Average Amount 
# Helps flag if a receiver suddenly gets a massive deposit compared to their norm
df['receiver_avg_amount'] = df.groupby('Receiver_account')['Amount'].transform('mean')
df['amount_to_receiver_avg_ratio'] = df['Amount'] / (df['receiver_avg_amount'] + 1e-5)

# Extra 2: Is Weekend Flag
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

# Extra 3: Time Since Last Sender Transaction (in seconds)
# Sudden bursts of transactions might have very low time differences
df = df.sort_values(by=['Sender_account', 'Datetime'])
df['time_since_last_txn_seconds'] = df.groupby('Sender_account')['Datetime'].diff().dt.total_seconds().fillna(-1)

# Drop redundant or intermediate columns to keep it clean (like the cumcount tracking logic)
df = df.drop(columns=['past_sender_receiver_count', 'country_risk_sender', 'country_risk_receiver'])

# Save the dataset
output_file = 'Feature_dataset.csv'
df.to_csv(output_file, index=False)
print("Dataset generated successfully. Shape:", df.shape)
print("Features created:", list(df.columns))

Dataset generated successfully. Shape: (101322, 34)
Features created: ['Datetime', 'Time', 'Date', 'Sender_account', 'Receiver_account', 'Amount', 'Payment_currency', 'Received_currency', 'Sender_bank_location', 'Receiver_bank_location', 'Payment_type', 'Is_laundering', 'Laundering_type', 'is_cross_border', 'currency_mismatch', 'transaction_hour', 'day_of_week', 'sender_txn_count', 'receiver_txn_count', 'sender_avg_amount', 'amount_deviation', 'amount_deviation_ratio', 'country_risk_score', 'sender_receiver_txn_count', 'transaction_velocity_24h', 'currency_risk', 'high_amount_flag', 'new_receiver', 'night_transaction', 'velocity_flag', 'receiver_avg_amount', 'amount_to_receiver_avg_ratio', 'is_weekend', 'time_since_last_txn_seconds']
